# 02 · Motivation (b) — Delay and gain heterogeneity in control logic

**Toy example.** Five agents follow the same stimulus $s(t)=\sin(0.5t)$ through a
linear controller with agent-specific *reaction delay* τᵢ and *gain* kᵢ:

$$ a_i(t) = -k_i\, s(t - \tau_i). $$

The global model (population mean) smears out timing and magnitude. The two-shot
decomposition recovers each agent, and — critically — the *inter-vehicle
disturbance* $|a_2(t) - a_1(t)|$, which the global model predicts as exactly zero.

**Figures produced:** `control_logic.png`, `Motivation_b_personalization.png`,
`Motivation_b_common.png`, `Motivation_b_unique.png`, `Motivation_b_disturbance.png`.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "bhd").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from sklearn.decomposition import PCA

from bhd import FIG_DIR, set_style

set_style()

## 1. Agent responses and the global (mean) model

In [ ]:
t = np.linspace(0, 20, 1000)
s = np.sin(0.5 * t)

n_agents = 5
tau = np.linspace(0.5, 2.0, n_agents)  # reaction delays
k = np.linspace(0.8, 2.0, n_agents)  # gains

responses = []
for i in range(n_agents):
    delayed_s = np.interp(t, t + tau[i], s, left=0, right=0)
    responses.append(-k[i] * delayed_s)

global_response = np.mean(responses, axis=0)
colors = cm.viridis(np.linspace(0, 1, n_agents))

plt.figure(figsize=(10, 7))
for i, response in enumerate(responses):
    plt.plot(t, response, color=colors[i], alpha=0.25, linewidth=7)
plt.plot([], [], color="black", linewidth=7, alpha=0.25, label="Vehicle i")
plt.plot(t, global_response, linestyle="-", color="black", linewidth=4, label="Global Model")
plt.xlabel("Time (sec)", fontsize=22)
plt.xticks(fontsize=20)
plt.ylabel("Acceleration Response", fontsize=22)
plt.ylim(-2.7, 2.2)
plt.yticks(fontsize=20)
plt.legend(fontsize=22, loc="lower center", ncol=3)
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "control_logic.png", dpi=300, bbox_inches="tight")
plt.show()

## 2. Two-shot decomposition

In [ ]:
D = np.column_stack(responses)  # (1000, 5)

# Shot 1 — shared component
pca_shared = PCA(n_components=1)
C = pca_shared.fit(D.T).components_.T
Psi = C.T @ D
D_shared = C @ Psi

# Shot 2 — personalized component
personalized, reconstructed = [], []
for v in range(n_agents):
    R_v = (D[:, v] - D_shared[:, v]).reshape(-1, 1)
    pca_personal = PCA(n_components=1)
    Phi_v = pca_personal.fit_transform(R_v)
    P_v = pca_personal.components_
    R_hat = Phi_v @ P_v
    personalized.append(R_hat.flatten())
    reconstructed.append(D_shared[:, v] + R_hat.flatten())

personalized_matrix = np.column_stack(reconstructed)

plt.figure(figsize=(10, 7))
for i in range(n_agents):
    plt.plot(t, D[:, i], color=colors[i], alpha=0.25, linewidth=7)
    plt.plot(t, reconstructed[i], linestyle=":", color=colors[i], linewidth=4)
plt.xlabel("Time (sec)", fontsize=22)
plt.ylabel("Acceleration Response", fontsize=22)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.ylim(-2.7, 2.2)
plt.grid(True)
plt.legend(["Vehicle i", "Personalized Prediction i"], fontsize=22, loc="lower center", ncol=2)
plt.tight_layout()
plt.savefig(FIG_DIR / "Motivation_b_personalization.png", dpi=300, bbox_inches="tight")
plt.show()

## 3. Common vs. personalized components

In [ ]:
plt.figure(figsize=(10, 7))
for i in range(n_agents):
    plt.plot(t, D_shared[:, i], color=colors[i], alpha=0.4, linewidth=7)
plt.xlabel("Time (sec)", fontsize=22)
plt.ylabel("Acceleration Response", fontsize=22)
plt.xticks(fontsize=20)
plt.yticks([-2, -1, 0, 1, 2], fontsize=20)
plt.ylim(-2, 2)
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "Motivation_b_common.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(10, 7))
for i in range(n_agents):
    plt.plot(t, personalized[i], linestyle="--", color=colors[i], alpha=0.4, linewidth=7)
plt.xlabel("Time (sec)", fontsize=22)
plt.ylabel("Acceleration Response", fontsize=22)
plt.xticks(fontsize=20)
plt.yticks([-2, -1, 0, 1, 2], fontsize=20)
plt.ylim(-2, 2)
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "Motivation_b_unique.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. Disturbance propagation between vehicle 1 and vehicle 2

The global model assigns the same response to every vehicle, so its predicted
inter-vehicle disturbance is identically zero.

In [ ]:
true_l2 = np.abs(D[:, 1] - D[:, 0])
global_matrix = np.tile(global_response.reshape(-1, 1), (1, n_agents))
global_l2 = np.abs(global_matrix[:, 1] - global_matrix[:, 0])
personalized_l2 = np.abs(personalized_matrix[:, 1] - personalized_matrix[:, 0])

plt.figure(figsize=(10, 7))
plt.plot(t, true_l2, label="True Behavior", color="red", alpha=0.25, linewidth=7)
plt.plot(t, global_l2, label="Global Model", linestyle="-", color="black", linewidth=4)
plt.plot(t, personalized_l2, label="Personalized Model", linestyle=":", color="midnightblue", linewidth=4)
plt.xlabel("Time (sec)", fontsize=22)
plt.ylabel("Disturbance Magnitude", fontsize=22)
plt.xticks(fontsize=20)
plt.yticks([0, 0.1, 0.2, 0.3, 0.4], fontsize=20)
plt.ylim(-0.12, 0.4)
plt.legend(loc="lower center", fontsize=22, ncol=2)
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "Motivation_b_disturbance.png", dpi=300, bbox_inches="tight")
plt.show()